# Checkpoint B2: JSON Response Schema

Day la schema hoan chinh va vi du validation.

**Cach su dung:** Chay tat ca cells de xem schema, thu validation voi mau hop le va loi.

In [ ]:
import json
from pathlib import Path

# Thu import jsonschema (khong co thi van chay duoc)
try:
    import jsonschema
    HAS_JSONSCHEMA = True
    print("jsonschema da san sang!")
except ImportError:
    HAS_JSONSCHEMA = False
    print("jsonschema chua cai dat. Se dung manual validation.")
    print("Cai dat: pip install jsonschema")

In [ ]:
# Load schema tu file
SKILL_DIR = Path("../../templates/skills/hr-policy-qa-skill")
schema_path = SKILL_DIR / "schemas" / "hr-response.schema.json"

with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

print(f"Da load schema: {schema.get('title', 'N/A')}")
print(f"Mo ta: {schema.get('description', 'N/A')}")
print(f"JSON Schema version: {schema.get('$schema', 'N/A')}")

In [ ]:
# In tom tat schema: required fields, types, enums
print("TOM TAT SCHEMA: hr-response.schema.json")
print("=" * 60)

# Required fields
required = schema.get("required", [])
print(f"\nRequired fields ({len(required)}):")
for field in required:
    print(f"  - {field}")

# Properties summary
properties = schema.get("properties", {})
print(f"\nProperties ({len(properties)}):")
print(f"{'Field':<25} {'Type':<15} {'Enum/Details':<30}")
print("-" * 70)

for field, prop in properties.items():
    ptype = prop.get("type", "N/A")
    enum_vals = prop.get("enum", None)
    desc = prop.get("description", "")[:50]
    
    if enum_vals:
        detail = f"enum: {enum_vals}"
    elif ptype == "array":
        items = prop.get("items", {})
        detail = f"array of {items.get('type', 'object')}"
    elif ptype == "number":
        mn = prop.get("minimum", "")
        mx = prop.get("maximum", "")
        detail = f"range: {mn}-{mx}"
    else:
        detail = desc[:30]
    
    req_mark = "*" if field in required else " "
    print(f"{req_mark} {field:<24} {ptype:<15} {detail}")

# Nested object: self_check_result
sc_prop = properties.get("self_check_result", {})
sc_required = sc_prop.get("required", [])
sc_props = sc_prop.get("properties", {})
print(f"\nself_check_result (nested object, {len(sc_required)} required):")
for f in sc_required:
    t = sc_props.get(f, {}).get("type", "?")
    print(f"    - {f}: {t}")

In [ ]:
# Tao mau response hop le (in-scope) va validate
valid_in_scope = {
    "question": "Nhan vien chinh thuc co bao nhieu ngay phep nam?",
    "classification": "in-scope",
    "answer": "Nhan vien chinh thuc cua VinaTel Network duoc huong ngay phep nam theo tham nien: duoi 5 nam duoc 12 ngay, tu 5 den duoi 10 nam duoc 14 ngay, tu 10 nam tro len duoc 16 ngay.",
    "citations": [
        {
            "doc_id": "POL-LEAVE-001",
            "section": "1. Ngh\u0129 phep nam -> 1.1 So ngay phep",
            "quote": "Nhan vien chinh thuc duoc huong ngay phep nam theo tham nien: Duoi 5 nam: 12 ngay; Tu 5 den duoi 10 nam: 14 ngay; Tu 10 nam tro len: 16 ngay.",
            "relevance_score": 0.95
        }
    ],
    "confidence": 0.95,
    "is_out_of_scope": False,
    "refusal_message": "",
    "self_check_result": {
        "passed": True,
        "issues_found": [],
        "corrected": False
    },
    "retrieval_method": "vector",
    "top_chunks_used": 1
}

print("MAU 1: In-scope response (hop le)")
print("=" * 60)

if HAS_JSONSCHEMA:
    try:
        jsonschema.validate(valid_in_scope, schema)
        print("VALIDATION: THANH CONG \u2713")
    except jsonschema.ValidationError as e:
        print(f"VALIDATION: THAT BAI \u2717")
        print(f"Loi: {e.message}")
else:
    # Manual validation
    missing = [f for f in required if f not in valid_in_scope]
    if not missing:
        print("VALIDATION (manual): THANH CONG \u2713")
        print("  Tat ca required fields deu co.")
    else:
        print(f"VALIDATION (manual): THAT BAI \u2717")
        print(f"  Thieu: {missing}")

print(f"\nCac truong co: {list(valid_in_scope.keys())}")

In [ ]:
# Tao mau response hop le (out-of-scope / refusal) va validate
valid_out_of_scope = {
    "question": "Cong ty dong bao hiem xa hoi bao nhieu phan tram?",
    "classification": "out-of-scope",
    "answer": "",
    "citations": [],
    "confidence": 0.0,
    "is_out_of_scope": True,
    "refusal_message": "Cau hoi cua ban ve bao hiem xa hoi nam ngoai pham vi kho tri thuc chinh sach nhan su hien tai. Toi chi ho tro tra loi ve: nghi phep, phu cap, tham nien va dao tao. Vui long lien he phong Nhan su (HR) de duoc ho tro.",
    "self_check_result": {
        "passed": True,
        "issues_found": [],
        "corrected": False
    },
    "retrieval_method": "none",
    "top_chunks_used": 0
}

print("MAU 2: Out-of-scope response (tu choi)")
print("=" * 60)

if HAS_JSONSCHEMA:
    try:
        jsonschema.validate(valid_out_of_scope, schema)
        print("VALIDATION: THANH CONG ✓")
    except jsonschema.ValidationError as e:
        print(f"VALIDATION: THAT BAI ✗")
        print(f"Loi: {e.message}")
else:
    missing = [f for f in required if f not in valid_out_of_scope]
    if not missing:
        print("VALIDATION (manual): THANH CONG ✓")
        print("  Tat ca required fields deu co.")
    else:
        print(f"VALIDATION (manual): THAT BAI ✗")
        print(f"  Thieu: {missing}")

oos_flag = valid_out_of_scope["is_out_of_scope"]
oos_refusal = valid_out_of_scope["refusal_message"][:80]
print(f"\nTu choi: is_out_of_scope={oos_flag}, answer=(trong)")
print(f"refusal_message: {oos_refusal}...")

In [ ]:
# Tao mau response LOI (thieu required field) va hien thi loi
invalid_response = {
    "question": "Nhan vien co duoc phu cap an trua khong?",
    "classification": "in-scope",
    "answer": "Nhan vien chinh thuc duoc phu cap an trua 30.000 VND/ngay.",
    # THIEU: citations (required!)
    "confidence": 0.85,
    "is_out_of_scope": False,
    "refusal_message": "",
    "self_check_result": {
        "passed": True,
        "issues_found": [],
        "corrected": False
    },
    "retrieval_method": "vector",
    "top_chunks_used": 2
    # THIEU: citations
}

print("MAU 3: Invalid response (thieu 'citations')")
print("=" * 60)

if HAS_JSONSCHEMA:
    try:
        jsonschema.validate(invalid_response, schema)
        print("VALIDATION: THANH CONG (bat ngo!)")
    except jsonschema.ValidationError as e:
        print("VALIDATION: THAT BAI \u2717 (dung nhu ky vong!)")
        print(f"Loi: {e.message}")
        print(f"Field bi loi: {list(e.path) if e.path else 'root'}")
        print(f"Validator: {e.validator}")
else:
    missing = [f for f in required if f not in invalid_response]
    if missing:
        print(f"VALIDATION (manual): THAT BAI \u2717 (dung nhu ky vong!)")
        print(f"Cac truong thieu: {missing}")
    else:
        print("VALIDATION (manual): THANH CONG (bat ngo!)")

print(f"\nCac truong co: {list(invalid_response.keys())}")
print(f"Cac truong thieu: {[f for f in required if f not in invalid_response]}")

In [ ]:
# Tong ket
print("=" * 60)
print("TONG KET CHECKPOINT B2")
print("=" * 60)
print(f"Schema co {len(required)} required fields")
print(f"  Required: {', '.join(required)}")
print(f"  Tong properties: {len(properties)}")
print()
print("Validation:")
print(f"  Mau in-scope (co citations): THANH CONG \u2713")
print(f"  Mau out-of-scope (tu choi):   THANH CONG \u2713")
print(f"  Mau loi (thieu citations):    THAT BAI \u2717")
print()
print(f"Ket luan: Schema co {len(required)} required fields, validation thanh cong voi 2 mau, that bai voi 1 mau loi.")
print()
print("Checkpoint B2 hoan thanh! Tiep tuc voi checkpoint-step-c1.ipynb.")